# Weather Forecast — Vinga A Station (SMHI)
Daily min/max temperature prediction using Machine Learning.

**Station:** Vinga A (71380) | **Lat:** 57.6322 | **Lon:** 11.6048  
**Data range:** 2016-02-01 → 2026-01-31  
**Target:** Predict next N days of min/max air temperature (°C)

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

RANDOM_STATE = 42
FORECAST_HORIZON = 30   # days ahead to forecast
DATA_FILE = "smhi-opendata_19_71380_201602_202601.csv"

## 2. Data Loading & Parsing

The SMHI file has a multi-line header (metadata rows) before the actual data.  
We skip those and parse the semicolon-separated temperature columns.

In [ ]:
def load_smhi(path: str) -> pd.DataFrame:
    """Parse SMHI open-data CSV with metadata header rows."""
    # Find the row where the actual data starts (contains 'Från Datum')
    with open(path, encoding="utf-8-sig") as f:
        lines = f.readlines()

    header_row = next(i for i, l in enumerate(lines) if l.startswith("Från Datum"))

    df = pd.read_csv(
        path,
        sep=";",
        skiprows=header_row,
        encoding="utf-8-sig",
        usecols=[2, 3, 5],   # representative date, t_min, t_max
    )

    df.columns = ["date", "t_min", "t_max"]
    df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
    df = df.dropna().sort_values("date").reset_index(drop=True)
    df["t_min"] = pd.to_numeric(df["t_min"], errors="coerce")
    df["t_max"] = pd.to_numeric(df["t_max"], errors="coerce")
    return df.dropna()


df = load_smhi(DATA_FILE)
print(f"Rows: {len(df)} | Date range: {df['date'].min().date()} → {df['date'].max().date()}")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Full temperature series
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df["date"], df["t_min"], color="steelblue", linewidth=0.6, label="T min")
axes[0].set_ylabel("°C")
axes[0].legend()
axes[1].plot(df["date"], df["t_max"], color="tomato", linewidth=0.6, label="T max")
axes[1].set_ylabel("°C")
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.suptitle("Daily Min/Max Temperature — Vinga A")
plt.tight_layout()
plt.show()

In [ ]:
# Monthly average + seasonal boxplot
df["month"] = df["date"].dt.month
monthly = df.groupby("month")[["t_min", "t_max"]].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
monthly.plot(ax=axes[0], marker="o", color=["steelblue", "tomato"])
axes[0].set_title("Average Monthly Temperature")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("°C")
axes[0].set_xticks(range(1, 13))

df.boxplot(column="t_max", by="month", ax=axes[1], color="tomato")
axes[1].set_title("T max distribution by month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("°C")
plt.suptitle("")
plt.tight_layout()
plt.show()

print(df[["t_min", "t_max"]].describe().round(2))

## 4. Feature Engineering

We create lag features, rolling statistics, and calendar features that give the model
information about recent temperature history and the seasonal cycle.

In [ ]:
def build_features(df: pd.DataFrame, lags: list[int], windows: list[int]) -> pd.DataFrame:
    feat = df.copy()

    # Calendar
    feat["day_of_year"] = feat["date"].dt.dayofyear
    feat["month"]       = feat["date"].dt.month
    feat["week"]        = feat["date"].dt.isocalendar().week.astype(int)
    feat["year"]        = feat["date"].dt.year

    # Fourier terms to encode annual seasonality (2 harmonics)
    for k in [1, 2]:
        feat[f"sin_{k}"] = np.sin(2 * np.pi * k * feat["day_of_year"] / 365.25)
        feat[f"cos_{k}"] = np.cos(2 * np.pi * k * feat["day_of_year"] / 365.25)

    # Lag features
    for lag in lags:
        feat[f"t_min_lag{lag}"] = feat["t_min"].shift(lag)
        feat[f"t_max_lag{lag}"] = feat["t_max"].shift(lag)

    # Rolling statistics (computed on lagged data to avoid leakage)
    for w in windows:
        feat[f"t_min_roll{w}_mean"] = feat["t_min"].shift(1).rolling(w).mean()
        feat[f"t_max_roll{w}_mean"] = feat["t_max"].shift(1).rolling(w).mean()
        feat[f"t_min_roll{w}_std"]  = feat["t_min"].shift(1).rolling(w).std()
        feat[f"t_max_roll{w}_std"]  = feat["t_max"].shift(1).rolling(w).std()

    # Temperature range as extra signal
    feat["t_range"] = feat["t_max"] - feat["t_min"]

    return feat.dropna()


LAGS    = [1, 2, 3, 7, 14]
WINDOWS = [7, 14, 30]

features = build_features(df, LAGS, WINDOWS)
print(f"Feature matrix: {features.shape}")
features.head(3)

## 5. Train / Test Split

We use a chronological split (no shuffling) — the last `FORECAST_HORIZON` days are the held-out test set.

In [ ]:
TARGET_COLS  = ["t_min", "t_max"]
DROP_COLS    = ["date", "t_min", "t_max", "t_range", "month"]  # month encoded via Fourier
FEATURE_COLS = [c for c in features.columns if c not in DROP_COLS]

X = features[FEATURE_COLS].values
y = features[TARGET_COLS].values
dates = features["date"].values

split_idx   = len(X) - FORECAST_HORIZON
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
dates_test       = dates[split_idx:]

print(f"Train: {split_idx} rows  |  Test: {len(X_test)} rows")
print(f"Test period: {pd.Timestamp(dates_test[0]).date()} → {pd.Timestamp(dates_test[-1]).date()}")

## 6. Model Training

Two models are trained side-by-side: Random Forest and Gradient Boosting.  
Both use `MultiOutputRegressor` to predict T min and T max simultaneously.

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300, max_depth=10, min_samples_leaf=5,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Gradient Boosting": MultiOutputRegressor(
        GradientBoostingRegressor(
            n_estimators=300, max_depth=5, learning_rate=0.05,
            subsample=0.8, random_state=RANDOM_STATE
        ),
        n_jobs=-1,
    ),
}

trained = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    print(f"  {name} trained.")

## 7. Evaluation on Test Set

In [ ]:
def evaluate(model, X, y_true, name=""):
    y_pred = model.predict(X)
    rows = []
    for i, col in enumerate(TARGET_COLS):
        mae  = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse = mean_squared_error(y_true[:, i], y_pred[:, i]) ** 0.5
        r2   = r2_score(y_true[:, i], y_pred[:, i])
        rows.append({"model": name, "target": col, "MAE": round(mae, 3),
                     "RMSE": round(rmse, 3), "R²": round(r2, 3)})
    return pd.DataFrame(rows), y_pred


results_list, preds = [], {}
for name, model in trained.items():
    res, pred = evaluate(model, X_test, y_test, name)
    results_list.append(res)
    preds[name] = pred

results = pd.concat(results_list, ignore_index=True)
results

In [ ]:
# Visualise actual vs predicted for both models
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
colors = {"Random Forest": "darkorange", "Gradient Boosting": "purple"}
date_index = pd.to_datetime(dates_test)

for ax, (target_idx, target_label, actual_color) in zip(
    axes,
    [(0, "T min (°C)", "steelblue"), (1, "T max (°C)", "tomato")]
):
    ax.plot(date_index, y_test[:, target_idx], label="Actual",
            color=actual_color, linewidth=1.5)
    for mname, pred in preds.items():
        ax.plot(date_index, pred[:, target_idx], label=mname,
                color=colors[mname], linestyle="--", linewidth=1.2)
    ax.set_ylabel(target_label)
    ax.legend(loc="upper left")

axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
fig.suptitle(f"Actual vs Predicted — last {FORECAST_HORIZON} days")
plt.tight_layout()
plt.show()

## 8. Feature Importance

Which signals matter most? (Random Forest native importance)

In [ ]:
rf_model = trained["Random Forest"]
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
top20 = importances.nlargest(20).sort_values()

top20.plot(kind="barh", figsize=(10, 7), color="steelblue")
plt.title("Top-20 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 9. Future Forecast

Generate a `FORECAST_HORIZON`-day forecast beyond the last known date using the best model.  
Because future lag values are unknown, we use **recursive forecasting** — each predicted day
feeds back as a lag feature for the next day.

In [ ]:
def recursive_forecast(model, history: pd.DataFrame, n_days: int,
                        lags: list[int], windows: list[int]) -> pd.DataFrame:
    """
    Iteratively predict one day at a time, appending each prediction to history
    so it can be used as a lag feature for the next step.
    """
    buf = history.copy()

    forecast_rows = []
    for _ in range(n_days):
        next_date = buf["date"].iloc[-1] + pd.Timedelta(days=1)

        # Build feature row for next_date
        feat_row = build_features(buf, lags, windows).iloc[[-1]][FEATURE_COLS]
        pred = model.predict(feat_row)[0]          # [t_min, t_max]

        forecast_rows.append({
            "date": next_date,
            "t_min": round(pred[0], 2),
            "t_max": round(pred[1], 2),
        })

        # Append prediction to buffer so next iteration can use it as a lag
        new_row = pd.DataFrame([{
            "date": next_date, "t_min": pred[0], "t_max": pred[1], "month": next_date.month
        }])
        buf = pd.concat([buf, new_row], ignore_index=True)

    return pd.DataFrame(forecast_rows)


# Pick the best model by average MAE across both targets
best_model_name = (
    results.groupby("model")["MAE"].mean().idxmin()
)
print(f"Best model: {best_model_name}")

forecast_df = recursive_forecast(
    trained[best_model_name], df.copy(), FORECAST_HORIZON, LAGS, WINDOWS
)
forecast_df.head()

In [ ]:
# Plot: last 60 days of history + forecast
context_days = 60
context = df.tail(context_days)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(context["date"], context["t_min"], context["t_max"],
                alpha=0.25, color="steelblue", label="Historical range")
ax.plot(context["date"], context["t_min"], color="steelblue", linewidth=1)
ax.plot(context["date"], context["t_max"], color="tomato", linewidth=1)

ax.fill_between(forecast_df["date"], forecast_df["t_min"], forecast_df["t_max"],
                alpha=0.35, color="orange", label="Forecast range")
ax.plot(forecast_df["date"], forecast_df["t_min"], "o--", color="darkorange",
        linewidth=1.2, markersize=4, label="Forecast T min")
ax.plot(forecast_df["date"], forecast_df["t_max"], "o--", color="red",
        linewidth=1.2, markersize=4, label="Forecast T max")

ax.axvline(df["date"].iloc[-1], color="gray", linestyle=":", linewidth=1.5,
           label="Last observed")
ax.set_ylabel("°C")
ax.set_title(f"{FORECAST_HORIZON}-day Temperature Forecast — {best_model_name}")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.tight_layout()
plt.show()

## 10. Export Forecast

Save the forecast table to CSV for downstream use.

In [ ]:
output_path = "forecast_output.csv"
forecast_df.to_csv(output_path, index=False)
print(f"Forecast saved to {output_path}")
forecast_df